<a href="https://colab.research.google.com/github/ronan-cunha/machine-learning-course/blob/main/Notebooks/Problem_set_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem set

## 1. Baixe a base de dados com os preços dos imóveis na Califórnia

from sklearn.datasets import fetch_california_housing

housing = fetch_california_housing()

df_real = pd.DataFrame(housing.data, columns=housing.feature_names)

df_real['MEDV'] = housing.target

## 2. Separe 20% das observações para a amostra de teste de forma aleatória.

## 3. Estime um modelo Ridge, Lasso e ElasticNet escolhendo de forma ótima os hiperparâmetros usando todas as as variáveis independentes disponíveis no dataset e a amostra de treino que foi separada no exerícicio anterior. Para cada modelo, descreva o conjunto de variáveis selecionadas ou mais importantes para cada modelo. Explique o motivo das variáveis selecionadas serem diferentes entre os modelos.

## 4. Dentre os modelos selecionados no exerícicio anterior, qual aquele com melhor MAPE e MPE calculado usando a amostra de teste.

# Solução

## 1. Baixe a base de dados com os preços dos imóveis na Califórnia

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import KFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso, ElasticNet, LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error
)


# 1. Carregamento dos dados
housing = fetch_california_housing()
X = pd.DataFrame(housing.data, columns=housing.feature_names)
y = housing.target

## 2. Separe 20% das observações para a amostra de teste de forma aleatória.

In [2]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

## 3. Estime um modelo Ridge, Lasso e ElasticNet escolhendo de forma ótima os hiperparâmetros usando todas as as variáveis independentes disponíveis no dataset e a amostra de treino que foi separada no exerícicio anterior. Para cada modelo, descreva o conjunto de variáveis selecionadas ou mais importantes para cada modelo. Explique o motivo das variáveis selecionadas serem diferentes entre os modelos.

In [3]:
class RegularizedRegressionCVModel:
    def __init__(self, model_type='ridge', inner_cv=5):
        """
        model_type: 'ridge', 'lasso', 'elasticnet'
        inner_cv: número de folds para a validação cruzada interna no conjunto de treino
        """
        self.model_type = model_type.lower()
        self.inner_cv = inner_cv
        self.best_params_ = None
        self.best_estimator_ = None
        pipe, param_grid = self._get_pipeline_and_params()

        inner_kf = KFold(n_splits=self.inner_cv,
                         shuffle=True,
                         random_state=42)
        self.grid_search = GridSearchCV(
            estimator=pipe,
            param_grid=param_grid,
            cv=inner_kf,
            scoring='neg_mean_squared_error',
            n_jobs=-1
        )

    def _get_pipeline_and_params(self):
        # Definição do Pipeline base (Scaler + Modelo)
        if self.model_type == 'ridge':
            pipe = Pipeline([('scaler', StandardScaler()),
                             ('model', Ridge())])
            param_grid = {'model__alpha': np.logspace(-3, 3, 30)}
        elif self.model_type == 'lasso':
            pipe = Pipeline([('scaler', StandardScaler()),
                             ('model', Lasso())])
            param_grid = {'model__alpha': np.logspace(-4, 1, 30)}
        elif self.model_type == 'elasticnet':
            pipe = Pipeline([('scaler', StandardScaler()),
                             ('model', ElasticNet())])
            param_grid = {
                'model__alpha': np.logspace(-4, 1, 15),
                'model__l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9]
            }
        else:
            raise ValueError("Tipo de modelo inválido. Escolha 'ridge', 'lasso' ou 'elasticnet'.")
        return pipe, param_grid

    def fit(self, X_train, y_train):
        # Ajusta e realiza o CV restrito à amostra de treino
        self.grid_search.fit(X_train, y_train)
        self.best_params_ = self.grid_search.best_params_
        self.best_estimator_ = self.grid_search.best_estimator_
        return self

    def predict(self, X_test):
        # Garantir que o modelo foi ajustado antes de tentar prever
        if self.best_estimator_ is None:
            raise RuntimeError("O modelo não foi ajustado. Chame fit() primeiro.")
        return self.best_estimator_.predict(X_test)

models_to_evaluate = [
    ('Ridge (L2)', 'ridge'),
    ('Lasso (L1)', 'lasso'),
    ('Elastic-Net', 'elasticnet')
]

best_models = []

for model_name, model_type in models_to_evaluate:

      # Instanciar e rodar o CV interno no treino
      regressor = RegularizedRegressionCVModel(model_type=model_type, inner_cv=5)
      regressor.fit(X_train, y_train)

      best_models.append({
            'Model_name': model_name,
            'Model': regressor,
            'Best_params': regressor.best_params_
        })

In [4]:
feature_names = X_train.columns
lasso_model = best_models[1]['Model'].best_estimator_.named_steps['model']
coefs_lasso = pd.Series(lasso_model.coef_, index=feature_names)
coefs_lasso

,0
MedInc,0.844869
HouseAge,0.120495
AveRooms,-0.294359
AveBedrms,0.342734
Population,-0.000000
AveOccup,-0.041385
Latitude,-0.890877
Longitude,-0.866618


In [5]:
feature_names = X_train.columns
ridge_model = best_models[0]['Model'].best_estimator_.named_steps['model']
coefs_ridge = pd.Series(ridge_model.coef_, index=feature_names)
coefs_ridge

,0
MedInc,0.846595
HouseAge,0.120345
AveRooms,-0.298771
AveBedrms,0.347206
Population,-0.000084
AveOccup,-0.041725
Latitude,-0.894315
Longitude,-0.870294


In [6]:
feature_names = X_train.columns
elasticnet_model = best_models[2]['Model'].best_estimator_.named_steps['model']
coefs_elasticnet = pd.Series(elasticnet_model.coef_, index=feature_names)
coefs_elasticnet

,0
MedInc,0.845503
HouseAge,0.120471
AveRooms,-0.295954
AveBedrms,0.344337
Population,-0.000000
AveOccup,-0.041517
Latitude,-0.891926
Longitude,-0.867755


In [7]:
coef_df = pd.DataFrame({
    'Lasso': coefs_lasso,
    'Ridge': coefs_ridge,
    'ElasticNet': coefs_elasticnet
})
coef_df

,Lasso,Ridge,ElasticNet
MedInc,0.844869,0.846595,0.845503
HouseAge,0.120495,0.120345,0.120471
AveRooms,-0.294359,-0.298771,-0.295954
AveBedrms,0.342734,0.347206,0.344337
Population,-0.000000,-0.000084,-0.000000
AveOccup,-0.041385,-0.041725,-0.041517
Latitude,-0.890877,-0.894315,-0.891926
Longitude,-0.866618,-0.870294,-0.867755


## 4. Dentre os modelos selecionados no exerícicio anterior, qual aquele com melhor MAPE e MPE calculado usando a amostra de teste

In [8]:
all_metrics = []

for i in range(len(best_models)):

      # Instanciar e rodar o CV interno no treino
      regressor = best_models[i]['Model']
      model_name = best_models[i]['Model_name']

      # Previsão no teste externo
      preds = regressor.predict(X_test)

      # Métricas de desempenho fora da amostra
      mse = mean_squared_error(y_test, preds)
      mae = mean_absolute_error(y_test, preds)
      mape = mean_absolute_percentage_error(y_test, preds)
      mpe = np.mean((y_test - preds) / y_test)

      all_metrics.append({
          'Model': model_name,
          'Best Params': str(regressor.best_params_) if regressor.best_params_ else 'N/A',
          'MSE': mse,
          'MAE': mae,
          'MAPE': mape,
          'MPE': mpe
      })

# 4. Agregação e Exibição dos Resultados
results_df = pd.DataFrame(all_metrics)
summary_df = results_df.groupby('Model')[['MSE', 'MAE', 'MAPE', 'MPE']].mean()

display(summary_df)

,MSE,MAE,MAPE,MPE
Model,,,,
Elastic-Net,0.536798,0.529577,0.317846,-0.124578
Lasso (L1),0.536708,0.529583,0.317839,-0.124703
Ridge (L2),0.536966,0.529571,0.317863,-0.124320
